# Rodada 32 — U-Net causal sobre os campos atmosféricos oficiais

Primeiro previsor base novo desde a S12. A rede lê os nove campos
atmosféricos oficiais e emite precipitação; nenhum mapa OOF de rodada
anterior entra como canal. Protocolo congelado em `experiments/ROUND32.md`.

Antes de rodar, confira em **Add Input** que os dois datasets estão
anexados (o oficial da competição e `worcap-round30-artefatos`, que agora
carrega também `ROUND32.md` e `round32_unet.py`), e em **Settings** que o
Accelerator é **GPU T4 x2** ou P100 e a Internet está ligada.

Tempo esperado: cerca de 2 horas de GPU para os seis blocos. `evaluate()`
pula blocos já salvos, então se a sessão cair basta rodar de novo.

## 1. Preencher os dois caminhos

In [ ]:
# Confira os nomes exatos no painel "Add Input" à direita.
ARTIFACTS_INPUT = "/kaggle/input/worcap-round30-artefatos"
COMPETITION_INPUT = "/kaggle/input/previsao-climatica-de-precipitacao-sobre-a-america-do-sul"

import os
print("artefatos:", os.listdir(ARTIFACTS_INPUT))
print("dados oficiais:", os.listdir(COMPETITION_INPUT))


## 2. Montar o layout do repositório em `/kaggle/working/repo`

Mesma estrutura relativa do repositório local, para que `ROOT` dentro de
`src/competition.py` resolva sozinho e o código rode sem modificação.

In [ ]:
import shutil
from pathlib import Path

REPO = Path("/kaggle/working/repo")
if REPO.exists():
    shutil.rmtree(REPO)
(REPO / "src").mkdir(parents=True)
(REPO / "data" / "raw").mkdir(parents=True)
(REPO / "experiments").mkdir(parents=True)

for py in Path(ARTIFACTS_INPUT, "code").glob("*.py"):
    shutil.copy2(py, REPO / "src" / py.name)

# Todos os protocolos vieram no bundle; `locked()` só precisa do hash.
for doc in Path(ARTIFACTS_INPUT, "code").glob("*.md"):
    shutil.copy2(doc, REPO / "experiments" / doc.name)
print("protocolos:", sorted(p.name for p in (REPO / "experiments").glob("*.md")))

dest_processed = REPO / "data" / "processed"
for sub in ("round9", "round10", "round20", "round25", "round27", "round28"):
    src_dir = Path(ARTIFACTS_INPUT, "artifacts", sub)
    if src_dir.exists():
        shutil.copytree(src_dir, dest_processed / sub, dirs_exist_ok=True)

# Sidecars de hash da S12: o bundle antigo não os trazia. Recria a partir
# do próprio .npy copiado, que é byte a byte o do repositório.
import hashlib, json
def sha256_of(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for npy_path in sorted((dest_processed / "round20").glob("*_s12.npy")):
    json_path = npy_path.with_suffix(".json")
    if not json_path.exists():
        json_path.write_text(json.dumps({"sha256": sha256_of(npy_path), "reference": "S12"}),
                             encoding="utf-8")
        print("sidecar recriado:", json_path.name)

print("Repositório montado em", REPO)


## 3. Ligar `data/raw` aos dados oficiais anexados

In [ ]:
import os

EXPECTED = [
    "treino_tp.nc", "treino_tp_alvo.nc", "teste_features.nc", "sample_submission.csv",
    "treino_t2.nc", "treino_cloud_cover.nc", "treino_shum_850.nc",
    "treino_surface_pressure.nc", "treino_u_850.nc", "treino_v_850.nc",
    "treino_temperature_850.nc", "treino_rel_hum_850.nc", "treino_geopotential_850.nc",
]
raw_dir = REPO / "data" / "raw"
missing = []
for name in EXPECTED:
    source = Path(COMPETITION_INPUT, name)
    target = raw_dir / name
    if source.exists():
        if not target.exists():
            os.symlink(source, target)
    else:
        missing.append(name)

print("faltando:", missing or "nada")


## 4. Reconstruir o cache oficial

`competition.audit()` regrava `tp.npy` **e os nove campos de variáveis**
(`t2.npy`, `u_850.npy`, ...) em `data/processed/official/`. A Rodada 32
depende desses nove arquivos — é a diferença em relação à Rodada 30, que só
precisava de `tp.npy`. São cerca de 2,9 GB; `/kaggle/working` comporta.

In [ ]:
%pip install -q netCDF4

import sys
sys.path.insert(0, str(REPO))

from src import competition
print("cache em:", competition.CACHE)
competition.audit()
print(sorted(p.name for p in competition.CACHE.glob("*.npy")))


## 5. Treinar a U-Net e decidir

`evaluate()` treina um modelo por corte causal e salva a previsão de grade
cheia em `data/processed/round32/`. Blocos já salvos são pulados, então
reexecutar depois de uma queda de sessão retoma de onde parou. `decision()`
compara com a S12 nas mesmas cinco janelas das rodadas 27 a 31.

In [ ]:
import torch
print("GPU:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(nenhuma)")

from src import round32_unet
round32_unet.evaluate()


In [ ]:
result = round32_unet.decision()
result


## 6. Salvar para trazer de volta ao repositório

In [ ]:
import shutil

OUT_DIR = Path("/kaggle/working/round32_output")
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True)
shutil.copytree(REPO / "reports" / "competition" / "round32", OUT_DIR / "reports",
                dirs_exist_ok=True)
shutil.copytree(REPO / "data" / "processed" / "round32", OUT_DIR / "oof",
                dirs_exist_ok=True)
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(OUT_DIR), p.stat().st_size // 1024, "KB")
print("\nPronto para baixar na aba Output.")
